# 02 Qwen2.5-72B-Instruct
Zero-shot Text-to-SQL on Spider dev with `qwen/qwen-2.5-72b-instruct` (temperature 0, 512 tokens).

In [2]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import display
from common import config, env, data, preflight
from common.config import RunConfig, resolve_limit, load_api_key
from common.llm_client import OpenRouterClient, fetch_endpoints
from common.pipeline import run_model_pipeline

In [3]:
# Run_Config (single source of truth for this notebook)
RUN_NAME = "qwen25_72b"
MODEL_ID = "qwen/qwen-2.5-72b-instruct"
LIMIT = resolve_limit(None)  # None = full dev set; or set A1_LIMIT
PROVIDER_ORDER = ("DeepInfra",) # e.g. ("DeepInfra",) -- pick from the fetch_endpoints table below

def make_cfg():
    return RunConfig(run_name=RUN_NAME, model_id=MODEL_ID, temperature=0.0, max_tokens=512,
                     reasoning_enabled=False, provider_order=tuple(PROVIDER_ORDER),
                     limit=LIMIT)

cfg = make_cfg()
cfg.to_dict()

{'run_name': 'qwen25_72b',
 'model_id': 'qwen/qwen-2.5-72b-instruct',
 'temperature': 0.0,
 'top_p': None,
 'top_k': None,
 'max_tokens': 512,
 'reasoning_enabled': False,
 'provider_order': ['DeepInfra'],
 'allow_fallbacks': False,
 'require_parameters': True,
 'seed': 0,
 'prompt_version': 'v1',
 'threads': 8,
 'limit': None,
 'max_attempts': 6,
 'request_timeout_s': 300.0,
 'preflight_n': 8,
 'retry_on_empty': False,
 'egsc_max_retries': 2,
 'egsc_exec_timeout_s': 5.0,
 'sample_rows': 3,
 'value_trunc': 64}

## Provider endpoints

In [4]:
# Available providers + reported quantization; choose PROVIDER_ORDER from this table.
endpoints = fetch_endpoints(MODEL_ID)
display(pd.DataFrame(endpoints))

,provider_name,quantization,context_length,pricing
0,DeepInfra,fp8,32768,"{'prompt': '0.00000036', 'completion': '0.0000..."
1,Novita,bf16,32000,"{'prompt': '0.00000038', 'completion': '0.0000..."


## Run, extract, predict, evaluate, Official_CLI check

In [5]:
client = OpenRouterClient(api_key=load_api_key(required=True))

In [6]:
cfg = make_cfg()
with env.track_run(RUN_NAME, LIMIT, extra={"fingerprint": cfg.fingerprint()}):
    res = run_model_pipeline(cfg, client, endpoints)
display(res.eval.table)
print("missing records:", res.n_missing, "| CLI check passed:", not res.mismatches)

qwen25_72b[full]:   0%|          | 0/1034 [00:00<?, ?it/s]

[pipeline] run: RunStats(n_total=1034, n_skipped=0, n_ok=1034, n_error=0, providers=Counter({'DeepInfra': 1034}), bad_lines=[], stopped_early=False)
[pipeline] wrapper totals match the Official_CLI.


,count,ex,em
easy,248,0.903226,0.705645
medium,446,0.878924,0.302691
hard,174,0.724138,0.293103
extra,166,0.584337,0.066265
all,1034,0.811412,0.359768


missing records: 0 | CLI check passed: True


## Provider report

In [7]:
# Served providers vs. the pin, with reported quantization (Req 12.5, 12.6, 22.5)
display(res.providers)
if (res.providers["flag"] == "outside pin").any():
    print("WARNING: some calls were served by providers outside the pin.")

,provider,calls,share,quantization,in_pin,flag
0,DeepInfra,1034,1.0,fp8,True,
